---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-47: Accessing Local and Remote MCP Servers from Python</h1>

# Learning agenda of this notebook

1. **Recap: Your Python Program Becomes the MCP Host**
    - Claude Desktop was the host in Lec-42 and Lec-44; today our own Python code is
    - Two ways to connect: **your code is the MCP client** (client-side), or **the LLM provider is the MCP client** (provider-side)
    - Setup: packages, keys and the Demo-server
2. **Connecting to a Local MCP Server with the FastMCP `Client`**
    - Connecting to our Demo-server over stdio
    - Each tool describes itself: input and output schemas
    - Calling tools, and the different views of a result
    - Reading resources and getting prompts
    - When a tool call fails
3. **Third-Party Local Servers and Many Servers at Once**
    - The **Git** server through `uvx`
    - The **Filesystem** server through `npx`, locked to a sandbox folder
    - Several servers at once with an `mcpServers` config (the same format as `claude_desktop_config.json`)
4. **Connecting to Remote MCP Servers over Streamable HTTP**
    - Our Demo-server on **Render**, with and without a token
    - The **Booking** server from Lec-44
    - **Hugging Face**'s hosted MCP server
    - Local vs. remote, side by side
5. **Under the Hood: The Same Connection with the Low-Level `mcp` SDK** (optional)
    - Transport → `ClientSession` → `initialize()`, and how each call maps to a JSON-RPC request
6. **Building an MCP-Powered Agent (Your Code is the MCP Client)**
    - Turning MCP tools into the `TOOL_SCHEMAS` an LLM understands
    - The Lec-41 agent loop with tools from an MCP server
    - **Capstone:** the Lec-41 coding agent rebuilt with its file tools coming from the Filesystem MCP server
7. **Letting the LLM Provider be the MCP Client (Hosted MCP Tool)**
    - **Groq** Responses API
    - **OpenAI** Responses API with approvals
    - **Anthropic** Messages API MCP connector
8. **Client-Side vs. Provider-Side MCP**
9. **Security and Troubleshooting**
10. **Key Takeaways and Check Your Understanding**

# <span style='background :lightgreen' >1. Recap: Your Python Program Becomes the MCP Host</span>

In **Lec-43** we met the three roles of the MCP architecture: the **host** (the AI application the user works with), the **MCP client** inside the host (one per server, it speaks the protocol), and the **MCP server** (exposes tools, resources and prompts). In Lec-42 and Lec-44 the host was always an app someone else wrote:

| | Lec-42 | Lec-44 | **Lec-45 (this notebook)** |
|---|---|---|---|
| **Host** | Claude Desktop | Claude Desktop / MCP Inspector | **Our own Python program** |
| **MCP client** | Built into Claude Desktop | Built into Claude Desktop / Inspector | **A few lines of Python** (`fastmcp.Client` or the `mcp` SDK) |
| **MCP servers** | GitHub, Jira, Gmail, Hugging Face, Filesystem, Git, Desktop Commander | Our **Demo-server** (local, then on Render) | **The same servers**, local and remote |
| **Who decides which tool to call** | Claude | Claude / you (in the Inspector) | **You** in code, or an **LLM** you call |

Today we write the host ourselves. There are two different ways to connect an LLM application to MCP servers, and this notebook covers both.

## a. Way 1: Your Code is the MCP Client (Client-Side MCP)

<div style="text-align:center;">
    <img src="../images/mcp-new-23.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

- **How it connects:** Your Python code → MCP client → MCP server. Your program opens the connection itself (it **spawns** a local server over stdio, or **calls** a remote one over HTTP), asks the server what it offers, and calls its tools.
- **What you can do once it's connected:**
    - **Local servers:** our Demo-server, the Git server and the Filesystem server from Lec-42, over stdio.
    - **Remote servers:** our Demo-server on Render and Hugging Face's hosted server, over Streamable HTTP.
    - **All three primitives:** tools, resources *and* prompts.
    - **Any LLM:** you can hand the tools to Groq, OpenAI, Claude or a local model, and build an agent (section 6).
- **Setting it up in 3 steps:** ① `async with Client(...)` → ② `await client.list_tools()` → ③ `await client.call_tool(name, args)`.

## b. Way 2: The LLM Provider is the MCP Client (Provider-Side MCP)

<div style="text-align:center;">
    <img src="../images/mcp-new-24.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

- **How it connects:** Your Python code → LLM provider → remote MCP server. You add one `mcp` entry to the `tools` of your API request, and the provider's cloud connects to the server, lists its tools and calls them while it generates the answer.
- **What you can do once it's connected:**
    - **No client code:** no connection to manage, just one extra entry in the request.
    - **Remote only:** the provider's cloud must reach the server, so it needs a public `https://…/mcp` URL. Local stdio servers cannot be used.
    - **Tools only:** resources and prompts are not supported.
    - **Approvals:** OpenAI and Groq can pause before each tool call and wait for your approval.
- **Setting it up in 3 steps:** ① Deploy the server at a public URL (Lec-44) → ② add `{"type": "mcp", ...}` to `tools` → ③ read the `mcp_call` items and the final answer (section 7).

## c. Setup
- Everything we need is already in the course environment: `fastmcp` (4.x), the official `mcp` SDK (2.x, installed with fastmcp), `openai`, `anthropic` and `python-dotenv`. **Node.js 22+** is needed for the `npx` servers in sections 3 and 6, and **uv** (from Lec-42) for the `uvx` server.
- Keys are read from `../keys/.env`: `GROQ_API_KEY`, `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `HF_TOKEN` (a free Hugging Face token from https://huggingface.co/settings/tokens) and `WEATHER_API_KEY` (used by the Demo-server).
- Run this notebook from the `Notebooks` folder. The Demo-server from Lec-44 lives in `mcp_demo_server/main.py`.
- **MCP clients are asynchronous**, so we write `await` before each call to a server and `async with` to open and close a connection. Jupyter lets us use `await` directly in a cell. (In a plain `.py` script, put the code in an `async def main()` and run it with `asyncio.run(main())`.)

In [1]:
import os, sys, json
from pathlib import Path
from importlib.metadata import version
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv("../keys/.env", override=True)
os.environ["FASTMCP_LOG_LEVEL"] = "WARNING"     # set before importing fastmcp: hides its informational log messages

for package in ["fastmcp", "mcp", "openai", "anthropic"]:
    print(f"{package:10} {version(package)}")
print("Demo-server found:", Path("mcp_demo_server/main.py").exists())

fastmcp    4.1.0
mcp        2.3.0
openai     2.54.0
anthropic  0.97.0
Demo-server found: True


# <span style='background :lightgreen' >2. Connecting to a Local MCP Server with the FastMCP <code>Client</code></span>

There are two Python libraries for writing an MCP client. This is the same choice we had in Lec-44 when we wrote the server (the low-level `Server` class vs. the `FastMCP` class), now on the client side:

- **The FastMCP `Client` (high-level), used in this notebook.** One class for every kind of server. It **infers the transport** from what you give it: a `Path` to a `.py` file → it spawns the server over stdio; a URL → Streamable HTTP; an `mcpServers` dictionary → several servers at once. `async with` connects, performs the handshake and disconnects for you, and results come back as convenient Python objects (`result.data`).
- **The official MCP Python SDK (`mcp` package, low-level).** You wire the transport, the session and the handshake together yourself. Section 5 shows the same connection written this way, so you can see what FastMCP does for you.

## a. Connecting to our Demo-server over stdio

- `Client(Path("mcp_demo_server/main.py"))` **starts** the Demo-server as a child process (with the same Python as this notebook) and talks to it over **stdio**, exactly as Claude Desktop did in Lec-44.
- Entering `async with` performs the handshake; leaving it shuts the server down. Each cell below opens its own short connection.
- `server_info` and `protocol_version` come from the handshake. FastMCP negotiates the newest protocol version both sides support.

In [2]:
from fastmcp import Client

demo_server = Path("mcp_demo_server/main.py")

async with Client(demo_server) as client:
    print("Connected to:", client.server_info.name, "| protocol version:", client.protocol_version)
    tools = await client.list_tools()

print(f"\nThe server offers {len(tools)} tools:")
for tool in tools:
    print(f"  - {tool.name:22} {tool.description}")

Connected to: Demo-server | protocol version: 2026-07-28

The server offers 7 tools:
  - add_numbers            Add two numbers together and returns their sum.
  - roll_dice              Roll n_dice 6-sided dice and return the result
  - get_datetime           Return the current date and time as a string.
  - fahrenheit_to_celsius  Convert a temperature from Fahrenheit to Celsius.
  - reverse_string         Reverse the given text string.
  - count_words            Count the number of words in a given text string.
  - get_weather            Fetch current weather info for a given city using WeatherAPI.com


## b. Each tool describes itself
FastMCP built these descriptions on the **server side** from the type hints and docstrings in `main.py` (Lec-44). The **input schema** is what an LLM needs to call the tool correctly. The **output schema** describes the structured result.

In [3]:
add_tool = next(tool for tool in tools if tool.name == "add_numbers")

print("Input schema: ", json.dumps(add_tool.input_schema, indent=2))
print("Output schema:", json.dumps(add_tool.output_schema, indent=2))

Input schema:  {
  "type": "object",
  "additionalProperties": false,
  "properties": {
    "a": {
      "type": "number"
    },
    "b": {
      "type": "number"
    }
  },
  "required": [
    "a",
    "b"
  ]
}
Output schema: {
  "properties": {
    "result": {
      "type": "number"
    }
  },
  "required": [
    "result"
  ],
  "type": "object",
  "x-fastmcp-wrap-result": true
}


## c. Calling tools
`call_tool(name, arguments)` sends a `tools/call` request and returns a `CallToolResult` with several views of the same answer:
- `result.data`: the result as a ready-to-use **Python object** (a `float`, a `list`, a `dict`, …).
- `result.content`: the raw MCP **content blocks** (here one `TextContent`). This text is what an LLM would see.
- `result.structured_content`: the result as **JSON**, matching the tool's output schema.
- `result.is_error`: `True` if the tool failed.

In [4]:
async with Client(demo_server) as client:
    result = await client.call_tool("add_numbers", {"a": 2, "b": 3})
    print("data:              ", result.data)
    print("content:           ", result.content)
    print("structured_content:", result.structured_content)
    print("is_error:          ", result.is_error)

    print("\nroll_dice   ->", (await client.call_tool("roll_dice", {"n_dice": 3})).data)
    print("get_weather ->", (await client.call_tool("get_weather", {"city": "Lahore"})).data)

data:               5.0
content:            [TextContent(type='text', text='5.0', annotations=None, meta=None)]
structured_content: {'result': 5.0}
is_error:           False

roll_dice   -> [3, 2, 4]
get_weather -> {'Country': 'Pakistan', 'City': 'Lahore', 'Latitude': 31.5497, 'Longitude': 74.3436, 'Temperature (C)': 33.6, 'Temperature (F)': 92.4, 'Condition': 'Sunny', 'Humidity (%)': 18, 'Wind (km/h)': 8.6, 'Feels Like (C)': 31.0, 'Last Updated': '2026-10-21 13:00'}


## d. Reading resources and getting prompts
Tools are only one of the three MCP primitives (Lec-43). The Demo-server also has two **resources** (read-only data, addressed by a URI) and one **prompt** (a reusable prompt template that the user, not the model, chooses).

In [5]:
async with Client(demo_server) as client:
    print("Resources:")
    for resource in await client.list_resources():
        print(f"  - {resource.uri}  ({resource.description})")
    contents = await client.read_resource("local://sample-data")
    print("\nContent of local://sample-data:\n" + contents[0].text)

    print("\nPrompts:")
    for prompt in await client.list_prompts():
        print(f"  - {prompt.name}({', '.join(arg.name for arg in prompt.arguments)}): {prompt.description}")
    rendered = await client.get_prompt("weather_report", {"city": "Karachi"})
    print("\nRendered prompt message:", rendered.messages[0].content.text)

Resources:
  - local://greeting  (A simple greeting message)
  - local://sample-data  (Sample student data)

Content of local://sample-data:
{
  "students": [
    "Hadeed",
    "Yashal",
    "Zalaid",
    "Miraal"
  ],
  "class": "AI Basics"
}

Prompts:
  - weather_report(city): Write a short, friendly weather report for a city.

Rendered prompt message: Use the get_weather tool to fetch the current weather in Karachi, then write a friendly three-sentence weather report.


## e. When a tool call fails
If the arguments do not match the input schema, the **server** rejects the call. By default `call_tool` raises a `ToolError`. With `raise_on_error=False` you get the result back with `is_error=True` instead, which is handy when an LLM chose the arguments and you want to send the error back to it.

In [6]:
from fastmcp.exceptions import ToolError

async with Client(demo_server) as client:
    try:
        await client.call_tool("add_numbers", {"a": "two"})
    except ToolError as error:
        print("ToolError:", str(error).splitlines()[0])

    result = await client.call_tool("add_numbers", {"a": "two"}, raise_on_error=False)
    print("is_error:", result.is_error)

ToolError: 2 validation errors for call[add_numbers]
is_error: True


# <span style='background :lightgreen' >3. Third-Party Local Servers and Many Servers at Once</span>

Any stdio server you configured for Claude Desktop in Lec-42 can be reached from Python. Instead of a `Path`, we give the client a **`StdioTransport`** with the same `command` and `args` you put in the config file.

- `log_file=` sends the server's log output (its stderr) to a file instead of the notebook. This also avoids a Windows-only crash, see section 9.
- `timeout=120`: the first run of `uvx` or `npx` downloads the server, which can take a while.

## a. The Git server through `uvx`
`uvx` downloads `mcp-server-git` from PyPI and runs it (Lec-42, section 7b). `--repository` locks the server to one repository: here, our course repository (the parent folder of `Notebooks`).

In [7]:
from fastmcp.client.transports import StdioTransport

course_repo = str(Path("..").resolve())
git_server = StdioTransport(command="uvx", args=["mcp-server-git", "--repository", course_repo],
                            log_file=Path("git-server.log"))

async with Client(git_server, timeout=120) as client:
    print("Git tools:", [tool.name for tool in await client.list_tools()], "\n")
    result = await client.call_tool("git_log", {"repo_path": course_repo, "max_count": 2})
    print(result.content[0].text)

Git tools: ['git_status', 'git_diff_unstaged', 'git_diff_staged', 'git_diff', 'git_commit', 'git_add', 'git_reset', 'git_log', 'git_create_branch', 'git_checkout', 'git_show', 'git_branch'] 

Commit history:
Commit: 1a67f0dff8c8c85ebf252e4a873f5cc48810e19c
Author: Arif Butt
Date: 2025-12-31 12:21:32+05:00
Message: Initial commit


Commit: 0fcf314956ce0cf2e332895a99b3e3187ae995dd
Author: Arif Butt
Date: 2025-12-31 12:12:41+05:00
Message: Initial Space app with Gradio




## b. The Filesystem server through `npx`, locked to a sandbox folder
The Filesystem server is a Node.js program, so `npx` downloads and runs it. First, check that Node.js (22 or newer) and `npx` are installed. If not, install Node.js from https://nodejs.org (LTS) and **restart Jupyter** so that it sees the new `PATH`.

In [8]:
!node --version
!npx --version

v22.22.3
10.9.8


We give the server a single **sandbox** folder. It can read and write files there, and **nowhere else**: the last call, which tries to read a system file, is refused by the server.

In [9]:
sandbox = Path("sandbox").resolve()                       # the only folder the server may touch
sandbox.mkdir(exist_ok=True)
files_server = StdioTransport(command="npx", args=["-y", "@modelcontextprotocol/server-filesystem", str(sandbox)],
                              log_file=Path("files-server.log"))

async with Client(files_server, timeout=120) as client:
    print("Filesystem tools:", [tool.name for tool in await client.list_tools()], "\n")

    result = await client.call_tool("write_file", {"path": str(sandbox / "hello.md"),
                                                   "content": "# Hello from Python\nWritten through an MCP server.\n"})
    print(result.content[0].text)
    result = await client.call_tool("list_directory", {"path": str(sandbox)})
    print("Sandbox contains:", result.content[0].text)

    try:
        await client.call_tool("read_text_file", {"path": "/etc/hosts"})
    except ToolError as error:
        print("Refused:", str(error)[:60], "...")

Filesystem tools: ['read_file', 'read_text_file', 'read_media_file', 'read_multiple_files', 'write_file', 'edit_file', 'create_directory', 'list_directory', 'list_directory_with_sizes', 'directory_tree', 'move_file', 'search_files', 'get_file_info', 'list_allowed_directories'] 

Successfully wrote to /Users/arif/Documents/genai-course/Notebooks/sandbox/hello.md
Sandbox contains: [FILE] hello.md
Refused: Access denied - path outside allowed directories: /etc/hosts ...


## c. Several Servers at Once with an `mcpServers` Config
A real host connects to several servers at the same time. FastMCP accepts a dictionary in **exactly the same shape as `claude_desktop_config.json`**, so you can reuse your Claude Desktop configuration. The client starts every server and merges their tools into one list. To keep names unique, each tool name gets the **server name as a prefix** (`demo_add_numbers`, `git_git_log`, `files_write_file`, …).

In [10]:
config = {
    "mcpServers": {
        "demo":  {"command": sys.executable, "args": ["mcp_demo_server/main.py"]},
        "git":   {"command": "uvx", "args": ["mcp-server-git", "--repository", course_repo]},
        "files": {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", str(sandbox)]},
    }
}

async with Client(config, timeout=120) as client:
    tool_names = [tool.name for tool in await client.list_tools()]
    print(f"{len(tool_names)} tools from 3 servers, for example:", tool_names[:3], tool_names[7:9], tool_names[-2:])
    print("demo_add_numbers ->", (await client.call_tool("demo_add_numbers", {"a": 40, "b": 2})).data)

33 tools from 3 servers, for example: ['demo_add_numbers', 'demo_roll_dice', 'demo_get_datetime'] ['git_git_status', 'git_git_diff_unstaged'] ['files_get_file_info', 'files_list_allowed_directories']
demo_add_numbers -> 42.0


# <span style='background :lightgreen' >4. Connecting to Remote MCP Servers over Streamable HTTP</span>

A remote server is not started by us: it is already running somewhere, and we connect to its URL. The code barely changes. We give the client a **URL instead of a path**, and, if the server needs one, a token with `auth=` (sent as the header `Authorization: Bearer <token>`).

## a. Our Demo-server on Render
In Lec-44 (section 4) we deployed the Demo-server to Render at `https://mcp-render-server-xzbh.onrender.com/mcp`, and the code below uses that URL. To use your own deployment instead, put its URL and, if you set one, the server's `MCP_API_TOKEN` in `../keys/.env`:
```
DEMO_SERVER_URL=https://<your-service>.onrender.com/mcp
MCP_API_TOKEN=<the same token you set on Render>
```
- **Testing locally?** Start the deployable server over HTTP, as in Lec-44 (`PORT=8000 uv run main.py` inside `mcp_render_server`), and use `DEMO_SERVER_URL=http://127.0.0.1:8000/mcp`. Section 7 needs a public `https` URL, so it skips its Demo-server examples in that case.
- A free Render service **sleeps** after 15 minutes without traffic, so the first request can take about a minute. That is why we use a generous `timeout`.

In [11]:
DEMO_SERVER_URL = os.getenv("DEMO_SERVER_URL", "https://mcp-render-server-xzbh.onrender.com/mcp")   # the Demo-server we deployed in Lec-44
DEMO_SERVER_TOKEN = os.getenv("MCP_API_TOKEN")           # None if your server has no token

if "<your-service>" in DEMO_SERVER_URL:
    print("Set DEMO_SERVER_URL in ../keys/.env first (see the cell above).")
else:
    async with Client(DEMO_SERVER_URL, auth=DEMO_SERVER_TOKEN, timeout=90) as client:
        print("Connected to:", client.server_info.name)
        print("Tools:", [tool.name for tool in await client.list_tools()])
        print("get_datetime ->", (await client.call_tool("get_datetime", {})).data, "(the server's clock)")
        print("add_numbers  ->", (await client.call_tool("add_numbers", {"a": 12.5, "b": 30.75})).data)

Connected to: Demo-server
Tools: ['add_numbers', 'roll_dice', 'get_datetime', 'fahrenheit_to_celsius', 'reverse_string', 'count_words', 'get_weather']
get_datetime -> 2026-10-21 08:11:55 (the server's clock)
add_numbers  -> 43.25


If the server is protected with `MCP_API_TOKEN`, a client without the token is turned away. This is the protection we added in Lec-44:

In [12]:
if DEMO_SERVER_TOKEN and "<your-service>" not in DEMO_SERVER_URL:
    try:
        async with Client(DEMO_SERVER_URL, timeout=90) as client:      # no auth= this time
            await client.list_tools()
    except Exception as error:
        print("Without the token:", type(error).__name__, "-", error)
else:
    print("This server has no token, so there is nothing to test here.")

This server has no token, so there is nothing to test here.


## b. The Booking Server from Lec-44
In Lec-44 (section 6) you connected Claude Desktop to a hotel-search server at `https://booking-mcp-server.onrender.com/mcp`. It needs **no token**. From Python we list its single tool and call it, with check-in two weeks from today.

- The server searches **Booking.com** live through a free API quota that the whole class shares, so run this cell only when you need it.
- Like our Demo-server, it runs on a free Render instance that sleeps when idle, so the first request can take 30 to 60 seconds.

In [13]:
from datetime import date, timedelta

BOOKING_URL = "https://booking-mcp-server.onrender.com/mcp"
check_in = date.today() + timedelta(days=14)              # two weeks from today
check_out = check_in + timedelta(days=2)

async with Client(BOOKING_URL, timeout=120) as client:
    print("Connected to:", client.server_info.name)
    for tool in await client.list_tools():
        print(f"  - {tool.name}: {tool.description[:110]}")
    result = await client.call_tool("search_hotels", {"city": "Lahore", "arrival_date": str(check_in),
                                                      "departure_date": str(check_out), "adults": 2})

print(f"\nHotels in Lahore from {check_in} to {check_out}:\n" + result.content[0].text[:900])

Connected to: hotel-price-server
  - search_hotels: 
Search REAL live hotel prices for a city and date range.

Args:
    city: the city to search, e.g. "Lahore"
 

Hotels in Lahore from 2026-11-04 to 2026-11-06:
{
  "ok": true,
  "city": "Lahore",
  "check_in": "2026-11-04",
  "check_out": "2026-11-06",
  "total_found": 20,
  "hotels": [
    {
      "name": "Hotel Swiss Pearl",
      "price": 25.2951536138332,
      "currency": "USD",
      "rating": 5.8,
      "review_count": 378,
      "stars": 2
    },
    {
      "name": "Al-Fateh Xpress Hotel",
      "price": 26.1400856760906,
      "currency": "USD",
      "rating": 6.4,
      "review_count": 87,
      "stars": 3
    },
    {
      "name": "Hotel Holiday",
      "price": 29.3379188283845,
      "currency": "USD",
      "rating": 5.3,
      "review_count": 330,
      "stars": 3
    },
    {
      "name": "Hotel The Leschanze",
      "price": 35.6246157201812,
      "currency": "USD",
      "rating": 7.2,
      "review_count": 551,

## c. Hugging Face's Hosted MCP Server
In Lec-42 you added Hugging Face to Claude Desktop as a **custom connector** with the URL `https://huggingface.co/mcp`. From Python, the same URL plus your `HF_TOKEN` is all it takes. The tools are written and hosted by Hugging Face; we only call them.

In [14]:
HF_MCP_URL = "https://huggingface.co/mcp"

async with Client(HF_MCP_URL, auth=os.getenv("HF_TOKEN"), timeout=60) as client:
    hf_tools = await client.list_tools()
    for tool in hf_tools:
        print(f"- {tool.name:18} {tool.description[:90]}...")

    result = await client.call_tool("hub_repo_search", {"author": "google", "repo_types": ["model"],
                                                        "sort": "downloads", "limit": 3})
print("\n" + result.content[0].text[:900], "...")

- hf_whoami          Inspect the current Hugging Face authentication context, including the account, visible or...
- hub_repo_search    Search Hugging Face repositories with a shared query interface. You can target models, dat...
- hub_repo_details   Get details for one or more Hugging Face repos (model, dataset, or space). Auto-detects ty...
- hf_fs              When to use: Hugging Face Hub models, datasets, Spaces, collections, papers, daily papers,...

Found 3 repositories across models matching author "google", sorted by downloads (descending).

## Models (3)

### google/electra-base-discriminator

**Library:** transformers | **Downloads:** 41.0M | **Likes:** 215

**Tags:** transformers, pytorch, tf, jax, rust, electra, pretraining, en, arxiv:1406.2661, license:apache-2.0, endpoints_compatible, region:us

**Created:** 2 Mar, 2022
**Link:** [https://hf.co/google/electra-base-discriminator](https://hf.co/google/electra-base-discriminator)

---

### google/vit-base-patch16-224

**Tas

**Local vs. remote, side by side.** Only the first argument of `Client(...)` changed:

| | Local (sections 2 and 3) | Remote (section 4) |
|---|---|---|
| **What you pass to `Client`** | `Path("main.py")`, a `StdioTransport`, or an `mcpServers` dict | A URL ending in `/mcp` |
| **Transport** | stdio | Streamable HTTP |
| **Who starts the server** | Your code, as a child process | Nobody: it is already running |
| **Authentication** | Not needed (it runs as you) | `auth="<token>"` (or OAuth) |
| **Everything after `async with`** | `list_tools()`, `call_tool()`, `read_resource()`, … | **Identical** |

# <span style='background :lightgreen' >5. Under the Hood: The Same Connection with the Low-Level <code>mcp</code> SDK (optional)</span>

The **official MCP Python SDK** (`mcp` package) is maintained by the Model Context Protocol project. It is what FastMCP is built on. You wire the pieces together yourself: a **transport** (`stdio_client` or `streamable_http_client`) that gives you a pair of read/write streams, a **`ClientSession`** on top of them, and an explicit **`initialize()`** handshake. It takes more code, but it shows every step of the protocol.

Every FastMCP call has a low-level twin, and each one is a single JSON-RPC request:

| What you want to do | FastMCP `Client` | Official `mcp` SDK (`ClientSession`) | JSON-RPC method (Lec-43) |
|---|---|---|---|
| Connect + handshake | `async with Client(...)` | `stdio_client(...)` + `ClientSession(...)` + `await session.initialize()` | `initialize` |
| List tools | `await client.list_tools()` | `await session.list_tools()` | `tools/list` |
| Call a tool | `await client.call_tool(name, args)` | `await session.call_tool(name, args)` | `tools/call` |
| List / read resources | `list_resources()` / `read_resource(uri)` | same names | `resources/list` / `resources/read` |
| List / get prompts | `list_prompts()` / `get_prompt(name, args)` | same names | `prompts/list` / `prompts/get` |

The last column is what you saw in the **Protocol** panel of the MCP Inspector in Lec-44: every Python call is one JSON-RPC request.

Here is the Demo-server again, this time with the official `mcp` package. Notice the extra layers that FastMCP hid from us:
1. **`StdioServerParameters`**: the command that starts the server, the same `command` + `args` you wrote in `claude_desktop_config.json`.
2. **`stdio_client(...)`**: starts the process and returns a pair of streams (`read`, `write`) connected to its stdin/stdout. `errlog=` says where the server's log output goes; we send it to the null device to keep the notebook tidy.
3. **`ClientSession(read, write)`**: the MCP session on top of the streams.
4. **`await session.initialize()`**: the handshake from the MCP life cycle (Lec-43). It returns the server's name, the agreed protocol version and the server's **capabilities**.

> In version 2 of the `mcp` SDK the result fields use Python-style names: `server_info`, `protocol_version`, `structured_content` (version 1 used `serverInfo`, `protocolVersion`, …). Many online examples still show the old names.

In [15]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(command=sys.executable, args=["mcp_demo_server/main.py"])

with open(os.devnull, "w") as devnull:
    async with stdio_client(server_params, errlog=devnull) as (read, write):
        async with ClientSession(read, write) as session:
            init = await session.initialize()
            print("Server:          ", init.server_info.name, init.server_info.version)
            print("Protocol version:", init.protocol_version)
            print("Capabilities:     tools =", init.capabilities.tools is not None,
                  "| resources =", init.capabilities.resources is not None,
                  "| prompts =", init.capabilities.prompts is not None)

            tools_result = await session.list_tools()
            print("Tools:           ", [tool.name for tool in tools_result.tools])

            result = await session.call_tool("add_numbers", {"a": 2, "b": 3})
            print("add_numbers ->   ", result.content[0].text, "| structured:", result.structured_content)

Server:           Demo-server 4.1.0
Protocol version: 2025-11-25
Capabilities:     tools = True | resources = True | prompts = True
Tools:            ['add_numbers', 'roll_dice', 'get_datetime', 'fahrenheit_to_celsius', 'reverse_string', 'count_words', 'get_weather']
add_numbers ->    5.0 | structured: {'result': 5.0}


- Same server, same tools, same answer. The low-level SDK negotiated protocol version `2025-11-25` with the classic `initialize` handshake, while the FastMCP `Client` above used the newer `2026-07-28` protocol. MCP clients and servers **negotiate** the version, so old and new code keep working together.
- Everything the FastMCP `Client` does maps one-to-one onto these low-level calls.

# <span style='background :lightgreen' >6. Building an MCP-Powered Agent (Your Code is the MCP Client)</span>

So far **we** decided which tool to call. Now an **LLM** decides, inside the agent loop you wrote in **Lec-41** (Examples 3 to 6). The loop itself does not change; only the place the tools come from:

| | Lec-41 | Lec-45 |
|---|---|---|
| **Where the tools live** | Python functions in the notebook (the `TOOLS` dictionary) | On an **MCP server** |
| **Describing the tools to the LLM** | Hand-written `TOOL_SCHEMAS` | Generated from `await mcp_client.list_tools()` |
| **Running a tool** | `TOOLS[name](**args)` | `await mcp_client.call_tool(name, args)` |
| **`run_tool()`, `run_agent()`, `main()`** | Ordinary functions | The same functions written with `async def` and `await`, because MCP calls are asynchronous |
| **The chat loop, the agent loop and the `"tool"` messages** | | **Identical** |

## a. Turning MCP tools into `TOOL_SCHEMAS`
Each MCP tool already carries everything the LLM needs, so the conversion is a direct mapping: the tool's `name` → `name`, its `description` → `description`, and its `input_schema` → `parameters`. Both examples below use this small function:

```python
def mcp_tools_to_schemas(mcp_tools):
    return [{"type": "function",
             "function": {"name": tool.name, "description": tool.description, "parameters": tool.input_schema}}
            for tool in mcp_tools]
```

## b. Example: The Lec-41 Agent Loop with Tools from our Demo-server
- `mcp_client = Client(Path("mcp_demo_server/main.py"))` gives the agent the Demo-server's seven tools. We wrote none of them in this notebook.
- `main()` opens the connection with `async with mcp_client:` **once**, so the server stays connected for the whole chat, and builds `TOOL_SCHEMAS` from `list_tools()`.
- `run_tool()` sends each call to the server with `call_tool(..., raise_on_error=False)`, so a failing tool returns its error as text, just like Lec-41's `"Error: ..."`.
- In a notebook we start the chat with `await main()` (Jupyter allows `await` at the top level of a cell). In a `.py` script, use `asyncio.run(main())`.

**Questions to ask** (type each one at the `You:` prompt, then type `exit` to finish the cell):
1. *Is it warmer in Lahore or in Islamabad right now?*
2. *Roll 3 dice and add up the numbers.*
3. *Reverse the string 'Model Context Protocol' and count its words.*

In [16]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from fastmcp import Client

load_dotenv('../keys/.env', override=True)
groq_api_key = os.getenv("GROQ_API_KEY")

client = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")


MODEL = "openai/gpt-oss-120b"                # Other Groq models with tool calling: openai/gpt-oss-20b, qwen/qwen3.8-27b

SYSTEM_PROMPT = """You are a helpful assistant. Use the tools whenever they help to answer the user's question,
then answer briefly."""


# ================================================================
# 1. The tools live on an MCP server: our Demo-server, started over stdio
# ================================================================
mcp_client = Client(Path("mcp_demo_server/main.py"))


def mcp_tools_to_schemas(mcp_tools):
    """Convert MCP tools into the function-tool format the LLM expects (the same TOOL_SCHEMAS format as Lec-41)."""
    return [{"type": "function",
             "function": {"name": tool.name, "description": tool.description, "parameters": tool.input_schema}}
            for tool in mcp_tools]


# ================================================================
# 2. Run one tool call: send it to the MCP server instead of calling a local function
# ================================================================
async def run_tool(tool_call):
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)
    print(f"  tool: {name}({args})")
    result = await mcp_client.call_tool(name, args, raise_on_error=False)   # Lec-41: TOOLS[name](**args)
    return result.content[0].text if result.content else "(no output)"


# ================================================================
# 3. The agent loop: exactly the Lec-41 loop, with `await` in front of run_tool()
# ================================================================
async def run_agent(messages):
    while True:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOL_SCHEMAS,
        )
        message = response.choices[0].message
        messages.append(message)

        # No tool calls means the model is done and answered in plain text
        if not message.tool_calls:
            return message.content

        for tool_call in message.tool_calls:
            result = await run_tool(tool_call)
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result,
            })


# ================================================================
# 4. The chat loop: connect to the server, ask it for its tools, then chat until the user types exit
# ================================================================
async def main():
    global TOOL_SCHEMAS
    async with mcp_client:                                              # starts the server; stays connected for the whole chat
        TOOL_SCHEMAS = mcp_tools_to_schemas(await mcp_client.list_tools())   # Lec-41: hand-written TOOL_SCHEMAS
        print(f"MCP agent ready with {len(TOOL_SCHEMAS)} tools from {mcp_client.server_info.name}. Type 'exit' to quit.")
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        while True:
            user_input = input("\nYou: ")
            if user_input.strip().lower() in ("exit", "quit"):
                break
            messages.append({"role": "user", "content": user_input})
            reply = await run_agent(messages)
            print(f"\nAgent: {reply}")


await main()                                                            # in a .py script: asyncio.run(main())

MCP agent ready with 7 tools from Demo-server. Type 'exit' to quit.

You: Is it warmer in Lahore or in Islamabad right now?
  tool: get_weather({'city': 'Lahore'})
  tool: get_weather({'city': 'Islamabad'})

Agent: Lahore is currently warmer, with a temperature of **33.6 °C (92.4 °F)**, compared to Islamabad’s **30.2 °C (86.4 °F)**.

You: Roll 3 dice and add up the numbers.
  tool: roll_dice({'n_dice': 3})
  tool: add_numbers({'a': 2, 'b': 6})
  tool: add_numbers({'a': 8, 'b': 3})

Agent: The three dice rolled [2, 6, 3] and their total is **11**.

You: Reverse the string 'Model Context Protocol' and count its words.
  tool: reverse_string({'text': 'Model Context Protocol'})
  tool: count_words({'text': 'locotorP txetnoC ledoM'})

Agent: Reversed string: **“locotorP txetnoC ledoM”**  
Word count: **3**.

You: exit


**What to notice**
- **Question 1:** the model asked for `get_weather` **twice**, once for each city, and only then compared them. A single round trip could not do this; the agent loop lets the model keep calling tools until it has everything it needs.
- **Question 2:** the model **chained** tools: `roll_dice`, then `add_numbers` to add the three results.
- The LLM had never seen these tools before. It chose them, and their arguments, from the names, descriptions and schemas that the **server** sent.
- **Swap the server, keep the code.** Replace `Client(Path(...))` with a URL (section 4) or an `mcpServers` config (section 3c), and the same agent gets Git, Filesystem, Booking or Hugging Face tools.

## c. Capstone: The Lec-41 Coding Agent, Rebuilt on an MCP Server
In Lec-41 (Example 6) we wrote all four tools of the coding agent ourselves: `list_files`, `read_file`, `write_file` and `run_command`. Here the **file tools come from the Filesystem MCP server** (section 3b), and only `run_command` stays our own function.

- **No file-tool code.** `list_directory`, `read_text_file`, `write_file` and `edit_file` are provided by the server.
- **The sandbox is enforced by the server.** It refuses any path outside `sandbox/` (section 3b). In Lec-41, `write_file` could overwrite any file in the notebook's folder. `run_command` runs inside the sandbox (`cwd=SANDBOX`) and still asks for your approval.
- **MCP tools and our own functions, side by side.** `run_tool()` first checks `LOCAL_TOOLS`; anything else goes to the MCP server. The model simply sees one list of tools.
- **Pick only the tools you need.** The Filesystem server offers 14 tools, but we pass only four (`MCP_TOOLS`). Every tool schema is resent on every call, so fewer tools means a shorter context (Lec-39) and keeps the agent within Groq's free limit of 8,000 tokens per minute (Lec-41, Example 6).
- **Real edits, not rewrites.** `edit_file` sends only the change (`oldText` → `newText`) instead of the whole file, the way real coding agents edit code.

**Questions to ask** (type each one at the `You:` prompt, approve the commands you are happy with at `Run '...'? [y/N]`, then type `exit`):
1. *What files are in the sandbox?*
2. *Write a Python script named fib.py that prints the first 10 Fibonacci numbers, then run it.*
3. *Add a short docstring to fib.py explaining what it does, and run it again.*

In [17]:
import os
import json
import platform
import subprocess
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from fastmcp import Client
from fastmcp.client.transports import StdioTransport

load_dotenv('../keys/.env', override=True)
groq_api_key = os.getenv("GROQ_API_KEY")

client = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")


MODEL = "openai/gpt-oss-120b"                # Other Groq models with tool calling: openai/gpt-oss-20b, qwen/qwen3.8-27b

SANDBOX = Path("sandbox").resolve()          # the only folder this agent may work in
SANDBOX.mkdir(exist_ok=True)

SYSTEM_PROMPT = f"""You are a coding agent running on the user's computer ({platform.system()}).
You work only inside the folder {SANDBOX}. Always pass full paths inside that folder to the file tools.
Use the file tools to list, read, write and edit files, and run_command to run commands (they run inside that folder).
Complete the user's task, then briefly summarize what you did."""


# ================================================================
# 1. File tools from an MCP server: the Filesystem server, locked to the sandbox folder
# ================================================================
files_server = StdioTransport(command="npx", args=["-y", "@modelcontextprotocol/server-filesystem", str(SANDBOX)],
                              log_file=Path("files-server.log"))
mcp_client = Client(files_server, timeout=120)
MCP_TOOLS = ["list_directory", "read_text_file", "write_file", "edit_file"]     # use only the tools this agent needs


# ================================================================
# 2. One tool of our own: run_command() from Lec-41, now running inside the sandbox
# ================================================================
def run_command(command):
    answer = input(f"  Run '{command}'? [y/N] ")          # Human-in-the-loop: nothing runs without a 'y'
    if answer.strip().lower() != "y":
        return "The user declined to run this command."
    result = subprocess.run(
        command, shell=True, cwd=SANDBOX, capture_output=True, text=True, errors="replace", timeout=120
    )
    output = (result.stdout + result.stderr).strip()
    return output or f"(no output, exit code {result.returncode})"


LOCAL_TOOLS = {
    "run_command": run_command,
}

LOCAL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "run_command",
            "description": "Run a shell command inside the sandbox folder and return its output. The user approves it first.",
            "parameters": {
                "type": "object",
                "properties": {
                    "command": {"type": "string", "description": "The shell command to run"},
                },
                "required": ["command"],
            },
        },
    },
]


def mcp_tools_to_schemas(mcp_tools):
    """Convert MCP tools into the function-tool format the LLM expects (the same TOOL_SCHEMAS format as Lec-41)."""
    return [{"type": "function",
             "function": {"name": tool.name, "description": tool.description, "parameters": tool.input_schema}}
            for tool in mcp_tools]


# ================================================================
# 3. Run one tool call: our own function, or a tool on the MCP server
# ================================================================
async def run_tool(tool_call):
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)
    print(f"  tool: {name}({args})")
    if name in LOCAL_TOOLS:                                                  # our own Python function
        try:
            return str(LOCAL_TOOLS[name](**args))
        except Exception as error:
            return f"Error: {error}"
    result = await mcp_client.call_tool(name, args, raise_on_error=False)   # a tool on the MCP server
    return result.content[0].text if result.content else "(no output)"


# ================================================================
# 4. The agent loop: exactly the Lec-41 loop, with `await` in front of run_tool()
# ================================================================
async def run_agent(messages):
    while True:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOL_SCHEMAS,
        )
        message = response.choices[0].message
        messages.append(message)

        # No tool calls means the model is done and answered in plain text
        if not message.tool_calls:
            return message.content

        for tool_call in message.tool_calls:
            result = await run_tool(tool_call)
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result,
            })


# ================================================================
# 5. The chat loop: connect to the server, pick its tools, add ours, then chat until the user types exit
# ================================================================
async def main():
    global TOOL_SCHEMAS
    async with mcp_client:
        mcp_tools = [tool for tool in await mcp_client.list_tools() if tool.name in MCP_TOOLS]
        TOOL_SCHEMAS = mcp_tools_to_schemas(mcp_tools) + LOCAL_SCHEMAS
        print(f"Mini coding agent ready with tools {[s['function']['name'] for s in TOOL_SCHEMAS]}. Type 'exit' to quit.")
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        while True:
            user_input = input("\nYou: ")
            if user_input.strip().lower() in ("exit", "quit"):
                break
            messages.append({"role": "user", "content": user_input})
            reply = await run_agent(messages)
            print(f"\nAgent: {reply}")


await main()                                                            # in a .py script: asyncio.run(main())

Mini coding agent ready with tools ['read_text_file', 'write_file', 'edit_file', 'list_directory', 'run_command']. Type 'exit' to quit.

You: What files are in the sandbox?
  tool: list_directory({'path': '/Users/arif/Documents/genai-course/Notebooks/sandbox'})

Agent: The sandbox directory contains a single file:

- `hello.md`

You: Write a Python script named fib.py that prints the first 10 Fibonacci numbers, then run it.
  tool: write_file({'content': 'def fib(n):\n    a, b = 0, 1\n    for _ in range(n):\n        yield a\n        a, b = b, a + b\n\nif __name__ == "__main__":\n    for num in fib(10):\n        print(num)\n', 'path': '/Users/arif/Documents/genai-course/Notebooks/sandbox/fib.py'})
  tool: run_command({'command': 'python3 fib.py'})
  Run 'python3 fib.py'? [y/N] y

Agent: I created `fib.py` in the sandbox with a simple function that generates Fibonacci numbers and a script block that prints the first 10 values. Then I executed the script with `python3 fib.py`, which outpu

**What to notice**
- For question 3 the model used `edit_file` with `oldText` and `newText`: only the changed lines were sent, not the whole file.
- Every path the model used was inside `sandbox/`. Ask it to read a file outside the sandbox, and the **server** refuses, whatever the model wants.
- Commands still needed your `y`: approval is our code's job, file safety is the server's job.
- **On Windows:** `run_command` runs in `cmd.exe`, and the system prompt tells the model the OS, so it uses `python` rather than `python3` (Lec-41, Example 6). If `npx` fails to start, see the Windows rows in section 9.

# <span style='background :lightgreen' >7. Letting the LLM Provider be the MCP Client (Hosted MCP Tool)</span>

In section 6 our code ran the tools. Several providers can do that work for us: we name a **remote** MCP server in the request, and the provider's cloud connects to it, lists its tools, calls them and writes the answer, all inside one API call. This is the *hosted tool* idea from Lec-40 (where the provider ran web search for us), applied to MCP.

We ask all three providers the same question, using Hugging Face's server (it is public and needs only your `HF_TOKEN`), and we also try our own Demo-server on Render.

In [18]:
from openai import OpenAI

groq = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=os.getenv("GROQ_API_KEY"))   # used by the Groq examples below

QUESTION = "Using Hugging Face, list the 3 most downloaded models published by the author google. Give names, download counts and links."
HF_TOKEN = os.getenv("HF_TOKEN")

def public_demo_server_url():
    """Our Demo-server can only be used here if it has a public https URL (a provider's cloud cannot reach localhost)."""
    if DEMO_SERVER_URL.startswith("https://") and "<your-service>" not in DEMO_SERVER_URL:
        return DEMO_SERVER_URL
    print(f"Skipped: provider-side MCP needs a public https URL, but DEMO_SERVER_URL is {DEMO_SERVER_URL!r}.")
    return None

## a. Groq Responses API
Groq supports remote MCP servers in its **Responses API** (the OpenAI-compatible API from Lec-40 and Lec-41), with any of its tool-calling models. The `mcp` tool takes:
- `server_label`: a short name for the server, shown in the output.
- `server_url`: the server's public URL.
- `headers`: sent only to that server. This is where the bearer token goes.
- `require_approval`: `"never"` lets Groq call tools on its own; `"always"` pauses for your approval (we try that with OpenAI below).

In [19]:
response = groq.responses.create(
    model="openai/gpt-oss-120b",
    input=QUESTION,
    tools=[{
        "type": "mcp",
        "server_label": "huggingface",
        "server_url": "https://huggingface.co/mcp",
        "headers": {"Authorization": f"Bearer {HF_TOKEN}"},
        "require_approval": "never",
    }],
)

print("Output items:", [item.type for item in response.output])
for item in response.output:
    if item.type == "mcp_call":
        print(f"  Groq called {item.name}({item.arguments})")
display(Markdown(response.output_text))

Output items: ['mcp_list_tools', 'reasoning', 'mcp_call', 'message']
  Groq called hub_repo_search({"author":"google","limit":3,"query":"","repo_types":["model"],"sort":"downloads"})


**Top 3 most‑downloaded models by the author “google” on Hugging Face**

| # | Model (link) | Downloads (approx.) | Brief description |
|---|--------------|--------------------|-------------------|
| 1 | **[google/electra‑base‑discriminator](https://hf.co/google/electra-base-discriminator)** | **41 M** | A BERT‑style discriminator from the ELECTRA pre‑training method (English). |
| 2 | **[google/vit‑base‑patch16‑224](https://hf.co/google/vit-base-patch16-224)** | **14 M** | Vision Transformer (ViT‑B/16) pretrained on ImageNet‑1k/21k for image classification. |
| 3 | **[google/gemma‑4‑26B‑A4B‑it](https://hf.co/google/gemma-4-26B-A4B-it)** | **10 M** | 4‑stage 26‑billion‑parameter Gemma model for image‑text‑to‑text tasks (multimodal). |

These figures come from a Hub search limited to the “google” namespace, sorted by download count and limited to the top three results.

**Reading the output items** (they are the same for Groq and OpenAI):

| Item | Meaning |
|---|---|
| `mcp_list_tools` | The provider connected to the server and fetched its tool list (the `tools/list` request). |
| `reasoning` | The model's reasoning step (reasoning models only). |
| `mcp_call` | One tool call made by the provider (the `tools/call` request), with `name`, `arguments`, `output` and `error`. |
| `mcp_approval_request` | Only with `require_approval`: the provider is waiting for your OK before calling a tool. |
| `message` | The final answer, also available as `response.output_text`. |

Now the same request against **our own Demo-server**. If the server has a token, it goes in the `Authorization` header:

In [20]:
server_url = public_demo_server_url()
if server_url:
    headers = {"Authorization": f"Bearer {DEMO_SERVER_TOKEN}"} if DEMO_SERVER_TOKEN else {}
    response = groq.responses.create(
        model="openai/gpt-oss-120b",
        input="Roll 3 dice for me, then tell me the total and what time it is on the server.",
        tools=[{"type": "mcp", "server_label": "demo_server", "server_url": server_url,
                "headers": headers, "require_approval": "never"}],
    )
    print([f"{item.name}({item.arguments})" for item in response.output if item.type == "mcp_call"])
    display(Markdown(response.output_text))

['roll_dice({"n_dice":3})', 'get_datetime({})']


I rolled three 6‑sided dice and got 1, 2, and 6. The total of the roll is **9**.

The server’s current date and time (UTC) is **2026‑10‑21 08:13:25**.

## b. OpenAI Responses API with Approvals
OpenAI's Responses API has the same `mcp` tool. Two options are worth knowing:
- **`allowed_tools`** imports only the tools you list. This keeps the prompt short and limits what the model can do.
- **`require_approval`** is `"always"` by default at OpenAI: the first response stops with an **`mcp_approval_request`** describing the call the model *wants* to make. You review it, then send an **`mcp_approval_response`** in a second request (linked with `previous_response_id`), and only then is the tool called. This is **human-in-the-loop** (Lec-39) built into the API.
- The server's token goes in the `authorization` field (OpenAI does not store it, so it is sent with every request).

In [21]:
client_openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
hf_tool = {
    "type": "mcp",
    "server_label": "huggingface",
    "server_url": "https://huggingface.co/mcp",
    "authorization": HF_TOKEN,
    "allowed_tools": ["hub_repo_search"],          # import only the tool we need
    "require_approval": "always",                   # pause before every tool call
}

# Request 1: the model plans a tool call and waits for approval
first = client_openai.responses.create(model="gpt-4o-mini", input=QUESTION, tools=[hf_tool])
print("Request 1 output items:", [item.type for item in first.output])
approval_requests = [item for item in first.output if item.type == "mcp_approval_request"]
for request in approval_requests:
    print(f"  The model wants to call {request.name} with {request.arguments}")

Request 1 output items: ['mcp_list_tools', 'mcp_approval_request']
  The model wants to call hub_repo_search with {"query":"","repo_types":["model"],"author":"google","filters":[],"sort":"downloads","limit":3}


The tool has **not** run yet. In a real app you would show this request to the user and ask *"Allow?"*. Here we approve it in code:

In [22]:
# Request 2: approve each pending call, continuing the same conversation
second = client_openai.responses.create(
    model="gpt-4o-mini",
    previous_response_id=first.id,
    tools=[hf_tool],
    input=[{"type": "mcp_approval_response", "approval_request_id": request.id, "approve": True}
           for request in approval_requests],
)
print("Request 2 output items:", [item.type for item in second.output])
display(Markdown(second.output_text))

Request 2 output items: ['mcp_call', 'message']


Here are the three most downloaded models published by the author **Google** on Hugging Face:

### 1. [google/electra-base-discriminator](https://hf.co/google/electra-base-discriminator)
- **Downloads:** 41.0M
- **Likes:** 215
- **Created:** 2 Mar, 2022

---

### 2. [google/vit-base-patch16-224](https://hf.co/google/vit-base-patch16-224)
- **Downloads:** 14.1M
- **Likes:** 1039
- **Created:** 2 Mar, 2022

---

### 3. [google/gemma-4-26B-A4B-it](https://hf.co/google/gemma-4-26B-A4B-it)
- **Downloads:** 10.1M
- **Likes:** 1689
- **Created:** 11 Mar, 2026

--- 

Feel free to check them out for more details!

> **Watch the arguments.** In our tests, when the question was vague (*"Urdu speech recognition models"*), OpenAI's models filled optional fields such as `author` with an empty string `""`. The Hugging Face server then found nothing, and the model reported that no models exist. Reviewing the `mcp_approval_request` catches exactly this kind of mistake, and specific prompts help.

## c. Anthropic Messages API MCP Connector
Anthropic's version is called the **MCP connector** (a beta feature, enabled with `betas=["mcp-client-2025-11-20"]`). It has two parts that must match by name:
- **`mcp_servers`**: *how to reach* each server (`url`, `name`, and an optional `authorization_token`).
- **`tools=[{"type": "mcp_toolset", "mcp_server_name": ...}]`**: *which tools* Claude may use from that server. Here we disable all tools by default and enable only `hub_repo_search` (an allowlist).

Claude's calls come back as **`mcp_tool_use`** and **`mcp_tool_result`** content blocks, next to the usual `text` blocks. The connector supports **tools only** (no resources or prompts) and **remote** servers only (Streamable HTTP or SSE).

In [23]:
import anthropic

claude = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

response = claude.beta.messages.create(
    model="claude-opus-5-5",
    max_tokens=16000,
    output_config={"effort": "low"},                # a simple lookup needs little thinking
    betas=["mcp-client-2025-11-20"],
    mcp_servers=[{"type": "url", "url": "https://huggingface.co/mcp", "name": "huggingface",
                  "authorization_token": HF_TOKEN}],
    tools=[{"type": "mcp_toolset", "mcp_server_name": "huggingface",
            "default_config": {"enabled": False},                    # allowlist: everything off ...
            "configs": {"hub_repo_search": {"enabled": True}}}],     # ... except this tool
    messages=[{"role": "user", "content": QUESTION}],
)

print("Content blocks:", [block.type for block in response.content])
for block in response.content:
    if block.type == "mcp_tool_use":
        print(f"  Claude called {block.server_name}.{block.name}({block.input})")
display(Markdown("".join(block.text for block in response.content if block.type == "text")))

Content blocks: ['mcp_tool_use', 'mcp_tool_result', 'text']
  Claude called huggingface.hub_repo_search({'author': 'google', 'sort': 'downloads', 'limit': 3, 'repo_types': ['model']})


These are Google's 3 most downloaded models on Hugging Face right now:

1. **google/electra-base-discriminator**: about 41.0M downloads. A text model that was pretrained but not fine-tuned for a specific task.
   [https://hf.co/google/electra-base-discriminator](https://hf.co/google/electra-base-discriminator)

2. **google/vit-base-patch16-224**: about 14.1M downloads. An image classification model.
   [https://hf.co/google/vit-base-patch16-224](https://hf.co/google/vit-base-patch16-224)

3. **google/gemma-4-26B-A4B-it**: about 10.1M downloads. A chat model that takes both images and text as input.
   [https://hf.co/google/gemma-4-26B-A4B-it](https://hf.co/google/gemma-4-26B-A4B-it)

Hugging Face rounds these counts, and they reflect recent downloads rather than all-time totals.

And our own Demo-server through the connector. Its token goes in `authorization_token`:

In [24]:
server_url = public_demo_server_url()
if server_url:
    server = {"type": "url", "url": server_url, "name": "demo_server"}
    if DEMO_SERVER_TOKEN:
        server["authorization_token"] = DEMO_SERVER_TOKEN
    response = claude.beta.messages.create(
        model="claude-opus-5-5", max_tokens=16000, output_config={"effort": "low"},
        betas=["mcp-client-2025-11-20"],
        mcp_servers=[server],
        tools=[{"type": "mcp_toolset", "mcp_server_name": "demo_server"}],      # all tools enabled
        messages=[{"role": "user", "content": "What is the weather in Karachi? Answer in two sentences."}],
    )
    display(Markdown("".join(block.text for block in response.content if block.type == "text")))

It's sunny in Karachi right now, at 28.6°C with 67% humidity. There's a 20.5 km/h wind, and this report was last updated on October 21, 2026 at 13:00.

# <span style='background :lightgreen' >8. Client-Side vs. Provider-Side MCP</span>

| | **Client-side** (sections 2–6) | **Provider-side** (section 7) |
|---|---|---|
| **Who is the MCP client** | Your Python code | The provider's cloud (Groq, OpenAI, Anthropic) |
| **Local stdio servers** | ✅ Yes | ❌ No: the server needs a public `https://` URL |
| **Primitives** | Tools, resources and prompts | Tools only |
| **Who runs the tool call** | Your code (`call_tool`) | The provider |
| **Which LLMs** | Any model that supports tool calling | Only providers that offer a hosted MCP tool |
| **Code you write** | Connection, schema conversion, tool loop | One extra entry in `tools` |
| **Control and approvals** | Full: you see and can block every call | `allowed_tools`, `require_approval` (OpenAI/Groq), allowlists (Anthropic) |
| **Who sees the tool results** | Your code, then the LLM | The provider and the LLM |
| **Best for** | Local or private servers, full control, agents (section 6) | Quick use of public remote servers with little code |

# <span style='background :lightgreen' >9. Security and Troubleshooting</span>

### Security
- **A server's tools run with real effect.** A local stdio server runs **as you**, with your files and permissions (Lec-42). Only connect servers you trust, and scope them, as we did with the Filesystem sandbox and the Git `--repository` option.
- **Everything a tool returns goes to the LLM.** Tool results become part of the prompt, and with provider-side MCP the provider sees them as well. A malicious server can also hide **prompt injections** in its tool descriptions or results.
- **Tokens belong in headers, never in URLs**, and in `../keys/.env`, never in the notebook. A public server without a token (Lec-44) can be used by anyone who finds its URL, at your cost.
- **Keep a human in the loop** for actions that change things: `require_approval`, `allowed_tools`, or simply reviewing the tool calls in your own code.

### Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| **Windows:** `io.UnsupportedOperation: fileno` when starting a stdio server in Jupyter | The server's log output (stderr) is sent to the notebook, which has no real file handle on Windows | Use a transport with `log_file=Path("server.log")` (FastMCP) or `errlog=` a real file (low-level SDK), as in sections 3 and 5 |
| **Windows:** a server started with `npx` fails, and the path contains spaces (`C:\Program Files\nodejs\npx.cmd`) | The command path is not quoted | Start it through the shell: `StdioTransport(command="cmd", args=["/c", "npx", "-y", "@modelcontextprotocol/server-filesystem", path])` |
| `uvx` / `npx` server times out on the first run | The package is being downloaded | Use `timeout=120`, or run the command once in a terminal first |
| `FileNotFoundError` / `spawn ... ENOENT` | `uvx`, `npx` or `node` is not on the `PATH` that Jupyter sees | Install it, then **restart Jupyter** (restarting the kernel is not enough); or give the absolute path from `which uvx` |
| `Client(...)` to a Render URL hangs for about a minute | The free Render instance was asleep | Wait, or open `https://<your-service>.onrender.com/health` in a browser first |
| `401` / *Server returned an error response* from our Demo-server | Missing or wrong token | `auth=` must be the same value as `MCP_API_TOKEN` on Render |
| Groq returns `424 Failed Dependency` (`external_connector_error`) | The provider could not reach or authenticate to the server | Check the URL is public `https`, the server is awake, and the header is right |
| The provider-side model answers *"nothing found"* | It called the tool with wrong or empty arguments | Inspect the `mcp_call` / `mcp_tool_use` arguments, make the prompt more specific, or use approvals |
| `AttributeError: ... has no attribute 'serverInfo'` | Code written for version 1 of the `mcp` SDK | Use the version 2 names: `server_info`, `protocol_version`, `structured_content` |

# <span style='background :lightgreen' >10. Key Takeaways and Check Your Understanding</span>

### Key takeaways
- In Lec-42 and Lec-44 the **host** was Claude Desktop. Today **our own Python program** was the host, and a few lines of code were its **MCP client**.
- The **FastMCP `Client`** infers the transport: a `Path` → stdio (local), a URL → Streamable HTTP (remote), an `mcpServers` dict → several servers at once, in the same format as `claude_desktop_config.json`. The official **`mcp` SDK** does the same with explicit layers: transport → `ClientSession` → `initialize()`.
- Every client method is one **JSON-RPC request** (`tools/list`, `tools/call`, `resources/read`, `prompts/get`, …), exactly what the MCP Inspector showed in Lec-44.
- **Local and remote servers look the same** once connected: only the first argument of `Client(...)` changes.
- An **MCP-powered agent** is the Lec-41 agent loop with two changes: `TOOL_SCHEMAS` come from `list_tools()`, and `run_tool()` calls `call_tool()`. Everything else is identical.
- An agent can **mix MCP tools with its own functions**, and should be given **only the tools it needs**: every tool schema costs tokens on every call.
- **Provider-side MCP** (Groq and OpenAI `type: "mcp"`, Anthropic's MCP connector) needs no client code, but works only with **public remote** servers and **tools** only. Approvals and allowlists keep you in control.
- **Next:** in **Lec-48** the OpenAI Agents SDK does all of this for you (`MCPServerStdio`, `MCPServerStreamableHttp`, `HostedMCPTool`).

### Check your understanding
1. In the Host–Client–Server architecture, which role did Claude Desktop play in Lec-44, and which role did our Python program play today?
2. What does `Client(...)` do differently when you pass it a `Path`, a URL, or a dictionary with an `mcpServers` key?
3. What is the difference between `result.data`, `result.content` and `result.structured_content`? Which one does an LLM see?
4. Why does each tool name get a prefix such as `demo_` or `git_` when you connect to several servers at once?
5. Name the four steps of the low-level SDK connection in section 5. Which one is the MCP handshake?
6. In section 6, which three fields of an MCP tool become the LLM's tool schema?
7. Compare the agent in section 6b with the Lec-41 agent. Which lines changed, and why are `run_tool()`, `run_agent()` and `main()` now `async`?
8. Why does `main()` keep the `async with mcp_client:` block open for the whole chat, instead of opening a connection for each tool call?
9. In the capstone, which tools come from the MCP server and which one is our own function? How does `run_tool()` decide where to send a call?
10. The Filesystem server offers 14 tools, but the capstone passes only four to the LLM. Give two reasons.
11. Your Demo-server works with `Client("http://127.0.0.1:8000/mcp")`. Why does the same URL fail in Groq's or Claude's `mcp` tool?
12. With OpenAI's `require_approval: "always"`, what are the two requests, and what links them together?
13. Give two reasons to prefer client-side MCP and two reasons to prefer provider-side MCP.
14. Name two security measures from this notebook that limit what an MCP server, or an agent, can do.